In [1]:
pip install -U transformers peft trl accelerate bitsandbytes datasets

In [2]:
import torch
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer


In [3]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

TRAIN_FILE = "/content/data/train.jsonl"
VAL_FILE   = "/content/data/val.jsonl"

OUTPUT_DIR = "/content/adapters"

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

LR = 2e-4
BATCH = 4
EPOCHS = 3


In [4]:
def format_sample(example):

    if example["input"].strip():
        text = (
            "### Instruction:\n"
            + example["instruction"]
            + "\n\n### Input:\n"
            + example["input"]
            + "\n\n### Response:\n"
            + example["output"]
        )
    else:
        text = (
            "### Instruction:\n"
            + example["instruction"]
            + "\n\n### Response:\n"
            + example["output"]
        )

    return {"text": text}


dataset = load_dataset(
    "json",
    data_files={
        "train": TRAIN_FILE,
        "validation": VAL_FILE
    }
)

dataset = dataset.map(
    format_sample,
    remove_columns=dataset["train"].column_names
)


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)


In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [7]:
model = prepare_model_for_kbit_training(model)


In [8]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)


In [9]:
model = get_peft_model(model, lora_config)


In [10]:

model.print_trainable_parameters()


trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [11]:
model.config.use_cache = False
model.gradient_checkpointing_enable()


In [12]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,

    gradient_accumulation_steps=1,
    num_train_epochs=EPOCHS,

    learning_rate=LR,

    fp16=False,
    bf16=False,

    logging_steps=10,
    save_steps=500,
    eval_steps=500,
    eval_strategy="steps",

    save_total_limit=2,
    report_to="none"
)


In [13]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,

    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],

    formatting_func=lambda x: x["text"],

    args=training_args
)


In [14]:
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss


TrainOutput(global_step=81, training_loss=0.6749747296174368, metrics={'train_runtime': 75.3923, 'train_samples_per_second': 4.298, 'train_steps_per_second': 1.074, 'total_flos': 195469489963008.0, 'train_loss': 0.6749747296174368})

In [17]:
ls /content/adapters/checkpoint-81


adapter_config.json        README.md              tokenizer.json
adapter_model.safetensors  rng_state.pth          trainer_state.json
chat_template.jinja        scheduler.pt           training_args.bin
optimizer.pt               tokenizer_config.json


In [18]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


('/content/adapters/tokenizer_config.json',
 '/content/adapters/chat_template.jinja',
 '/content/adapters/tokenizer.json')

In [20]:
ls /content

adapters/  data/  __init__.py  sample_data/


In [21]:
ls /content/adapters


adapter_config.json        checkpoint-81/  tokenizer_config.json
adapter_model.safetensors  __init__.py     tokenizer.json
chat_template.jinja        README.md


In [24]:
from google.colab import files
files.download("llm_artifacts.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>